In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# 데이터셋 디렉토리 설정 = 35개 강아지/고양이 각각
train_dir = '../dataset/small_train'

# ImageDataGenerator를 사용하여 데이터 로드 및 전처리 -> train_datagen 객체
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,  # 이미지 픽셀 값을 0-1 범위로 스케일링
    shear_range=0.2,  # 랜덤한 전단 변환 적용
    zoom_range=0.2,  # 랜덤한 확대/축소 변환 적용
    horizontal_flip=True,  # 랜덤하게 이미지를 좌우 반전
    validation_split=0.2  # 80% 학습, 20% 검증 데이터로 분리
)

# 학습 데이터 생성기
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),  # 모든 이미지 크기를 224x224로 조정
    batch_size=32,  # 배치 크기
    class_mode='binary',  # 이진 분류를 위해 라벨을 이진 형식으로 설정
    subset='training'  # 학습 데이터 서브셋
)

# 검증 데이터 생성기
validation_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),  # 모든 이미지 크기를 224x224로 조정
    batch_size=32,  # 배치 크기
    class_mode='binary',  # 이진 분류를 위해 라벨을 이진 형식으로 설정
    subset='validation'  # 검증 데이터 서브셋
)

# MobileNetV2 모델 불러오기 (사전 학습된 가중치 사용, 최상위 레이어 제거)
# 모델의 최상위 레이어(분류 레이어)를 제외. 사전 학습된 부분은 이미지 특징 추출에만 사용
# 사용자 정의 분류 레이어를 추가할 수 있게됨 
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# 모델 구성
# MobileNetV2의 마지막 출력층을 받아옴. 이 출력은 다양한 이미지 특징들이 추출된 형태
x = base_model.output
# Global Average Pooling 레이어는 특징 맵의 공간 차원을 평균으로 압축
# 원을 줄이고 과적합을 방지 (추출된 특징을 더 간결하게 요약하여 다음 레이어에 전달)
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x) # 비선형성 추가, 특징 맵의 고급화
predictions = Dense(1, activation='sigmoid')(x)

# 최종 모델 구성
model = Model(inputs=base_model.input, outputs=predictions)

# 기본 모델의 레이어를 고정 (학습되지 않도록 설정)
for layer in base_model.layers:
    layer.trainable = False

# 모델 컴파일
model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

# 모델 학습
model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // validation_generator.batch_size,
    epochs=10  # 에포크 수
)

# 모델 저장
model.save('../models/model_small_train/cat_dog_classifier.keras')

Found 56 images belonging to 2 classes.
Found 14 images belonging to 2 classes.


Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 52s 52s/step - accuracy: 0.2500 - loss: 0.8568 - val_accuracy: 0.5000 - val_loss: 0.6694
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - accuracy: 0.5938 - loss: 0.6731 - val_accuracy: 0.8571 - val_loss: 0.4826
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - accuracy: 1.0000 - loss: 0.3973 - val_accuracy: 1.0000 - val_loss: 0.3599
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - accuracy: 0.9688 - loss: 0.4320 - val_accuracy: 1.0000 - val_loss: 0.2594
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step - accuracy: 1.0000 - loss: 0.2720 - val_accuracy: 1.0000 - val_loss: 0.2509
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 1.0000 - loss: 0.2262 - val_accuracy: 1.0000 - val_loss: 0.1937
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 1.0000 - loss: 0.1622 - val_accuracy: 1.0000 - val_loss: 0.1438
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 1.0000 - loss: 0.1392 - val_accuracy: 1.0000 - val_loss: 0.1236
Epoch 9/10
1/1